# Final Test Evaluation

One-time evaluation on the held-out test set, run only after all model and hyperparameter
decisions are final. Every model is fitted on the training set (tuned models choose their
hyperparameters with inner CV on the training set only) and scored once on the test set.

Uncertainty comes from 2,000 participant-level bootstrap resamples of the test set. Every model
uses the same resamples.

In [ ]:
import pandas as pd

import mxmh_common as mc

Y_test, predictions, selected = mc.run_test_evaluation()
MODEL_ORDER = list(mc.make_models().keys())
print(f"Test set: {len(Y_test)} participants")

## Selected Hyperparameters

Hyperparameters chosen by the inner CV on the full training set.

In [ ]:
selected

## Test-Set Performance

Estimate with the bootstrap 95% CI for R², RMSE and MAE.

In [ ]:
def fmt_ci(estimate, ci_low, ci_high, digits):
    return f"{estimate:.{digits}f} [{ci_low:.{digits}f}, {ci_high:.{digits}f}]"


boot = []
for target in mc.TARGETS:
    b = mc.bootstrap_test_metrics(Y_test[target], predictions[target])
    b.insert(0, "target", target)
    boot.append(b)
boot = pd.concat(boot, ignore_index=True)

DIGITS = {"R2": 3, "RMSE": 2, "MAE": 2}
rows = []
for (target, model), group in boot.groupby(["target", "model"]):
    by_metric = group.set_index("metric")
    row = {"Target": target, "Model": model}
    for metric, digits in DIGITS.items():
        m = by_metric.loc[metric]
        row[metric] = fmt_ci(m.estimate, m.ci_low, m.ci_high, digits)
    rows.append(row)
test_perf = pd.DataFrame(rows)
test_perf["Model"] = pd.Categorical(test_perf.Model, MODEL_ORDER, ordered=True)
test_perf = test_perf.sort_values(["Target", "Model"])
test_perf.to_csv(mc.RESULTS_DIR / "test_set_results_table.csv", index=False)
test_perf.set_index(["Target", "Model"])